# 304 — Orchestrator

End-to-end investigation flows driven by a single natural-language query.

The `Orchestrator` is a controller, not an agent. It:

1. Calls the **LLM planner** to classify the query and generate a plan
2. **Validates** MCP-registered plan steps via `validate_plan`
3. **Resolves** each entity to its canonical graph name via `resolve_entity`
4. **Executes** each plan step through `GraphAgent`, `RiskAgent`, or `TraceAgent`
5. **Enforces stop conditions** via `evaluate_stop_conditions` when present
6. **Creates and finalises a Neo4j trace** — every run is auditable

```
query
  └─ Planner          → PlannerResult (mode, entities, plan)
       └─ Trace       → InvestigationTrace persisted to Neo4j
       └─ validate_plan (MCP)
       └─ resolve_entity (MCP)  → canonical names
       └─ GraphAgent.run(task, context, trace)
       └─ RiskAgent.run(task, context, trace)
       └─ TraceAgent.run(task, context, trace)
       └─ evaluate_stop_conditions (MCP)
  └─ OrchestratorResult
       ├─ mode, query, trace_id, success
       ├─ planner_output
       ├─ resolved_entities
       ├─ step_results   (per-step success, summary, findings)
       ├─ final_answer
       └─ warnings / errors
```

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

import json
import uuid

from src.config import get_neo4j_settings, get_anthropic_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tracing.trace_service import TraceService
from src.clients.mcp_tool_client import MCPToolClient
from src.clients.anthropic_client import AnthropicClient
from src.agents.graph_agent import GraphAgent
from src.agents.risk_agent import RiskAgent
from src.agents.trace_agent import TraceAgent
from src.orchestration.planner import InvestigationPlanner
from src.orchestration.orchestrator import Orchestrator, OrchestratorResult

# Neo4j — used by TraceService and TraceRepository
neo4j_settings = get_neo4j_settings()
neo4j = Neo4jRepository(**vars(neo4j_settings))
neo4j.test_connection()
trace_repo = TraceRepository(neo4j)
trace_service = TraceService(trace_repo)

# AI client (Haiku by default)
anthropic_settings = get_anthropic_settings()
ai_client = AnthropicClient(anthropic_settings)
print(f"AI client: {anthropic_settings.model_haiku}")

# MCP client — lazy Neo4j connection via the server layer
mcp = MCPToolClient()
print(f"MCP client: {len(mcp.list_tools())} tools registered")

# Agents
graph_agent = GraphAgent(mcp, trace_service, ai_client)
risk_agent  = RiskAgent(mcp, trace_service, ai_client)
trace_agent = TraceAgent(mcp, trace_service, ai_client)

# Planner + Orchestrator
planner      = InvestigationPlanner(ai_client)
orchestrator = Orchestrator(
    planner=planner,
    mcp=mcp,
    graph_agent=graph_agent,
    risk_agent=risk_agent,
    trace_agent=trace_agent,
    trace_service=trace_service,
    trace_repo=trace_repo,
    ai_client=ai_client,
)
print("Orchestrator ready")

AI client: claude-haiku-4-5-20251001
MCP client: 15 tools registered
Orchestrator ready


## Display helpers

In [2]:
def show(label: str, r: OrchestratorResult) -> None:
    """Print a structured summary of an OrchestratorResult."""
    sep = "═" * 65
    print(f"\n{sep}")
    print(f"  {label}")
    print(sep)
    print(f"  Query        : {r.query}")
    print(f"  Mode         : {r.mode}")
    print(f"  Success      : {r.success}")
    print(f"  Trace ID     : {r.trace_id}")

    # Planner output
    print()
    print("  ── Planner ──────────────────────────────────────────────")
    po = r.planner_output
    print(f"  Reason       : {po.get('reason', '')}")
    for e in po.get("entities", []):
        print(f"  Entity       : {e['name']}  ({e['type']})")
    for s in po.get("plan", []):
        print(f"  Plan step    : {s['step_id']}  [{s['agent']}]  {s['task']}")

    # Resolved entities
    if r.resolved_entities:
        print()
        print("  ── Resolved entities ────────────────────────────────────")
        for orig, data in r.resolved_entities.items():
            if data:
                canonical = data.get("canonical_name", orig)
                exact     = data.get("exact_match", "?")
                print(f"  {orig!r:<30} → {canonical!r}  (exact={exact})")
            else:
                print(f"  {orig!r:<30} → NOT RESOLVED")

    # Step results
    print()
    print("  ── Steps ────────────────────────────────────────────────")
    for sr in r.step_results:
        if sr.status == "skipped":
            status_str = f"SKIPPED  ({sr.skip_reason})"
        elif sr.status == "failed":
            status_str = f"FAILED   ({sr.error})"
        else:
            status_str = sr.status.upper()
        tools = ", ".join(sr.tools_executed) if sr.tools_executed else "—"
        summary = (sr.summary[:80] + "…") if len(sr.summary) > 80 else sr.summary
        print(f"  {sr.step_id}  [{sr.agent}]  {sr.task:<35}  {status_str}")
        print(f"       tools: {tools}")
        if summary and sr.status == "success":
            print(f"       └─ {summary}")

    # Final answer
    print()
    print("  ── Final answer ─────────────────────────────────────────")
    answer = r.final_answer
    # Wrap at 60 chars
    while len(answer) > 60:
        print(f"  {answer[:60]}")
        answer = answer[60:]
    if answer:
        print(f"  {answer}")

    # Warnings / errors
    if r.warnings:
        print()
        for w in r.warnings:
            print(f"  ⚠  {w}")
    if r.errors:
        print()
        for e in r.errors:
            print(f"  ✗  {e}")
    print(sep)


# Collect trace IDs for cleanup at the end
trace_ids: list[str] = []

## Test 1 — Ownership only

Expected flow: `entity_lookup` → `expand_ownership`

Expected planner: mode=investigate, 2 steps, graph-agent only.

In [3]:
r1 = orchestrator.run(
    query="who owns VODAFONE 2.?",
    user_id="analyst-304",
)
trace_ids.append(r1.trace_id)
show("Test 1 — Ownership only", r1)

# Ownership findings
ownership_step = next((s for s in r1.step_results if s.task == "expand_ownership"), None)
if ownership_step and ownership_step.success:
    data  = ownership_step.findings.get("expand_ownership") or {}
    paths = data.get("paths", []) if data else []
    ubos  = data.get("ultimate_owners", []) if data else []
    print(f"\nOwnership hops : {len(paths)}")
    print(f"UBOs found     : {len(ubos)}")

[03/23/26 19:25:47] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=206353;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=31212;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:25:49] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=219638;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=182632;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


═════════════════════════════════════════════════════════════════
  Test 1 — Ownership only
═════════════════════════════════════════════════════════════════
  Query        : who owns VODAFONE 2.?
  Mode         : investigate
  Success      : True
  Trace ID     : 80097453-0625-40fe-9455-b05583e6de69

  ── Planner ──────────────────────────────────────────────
  Reason       : Ownership query for a company. Resolving the canonical name first, then walking the full ownership chain to identify all owners.
  Entity       : VODAFONE 2.  (Company)
  Plan step    : step_1  [graph-agent]  entity_lookup
  Plan step    : step_2  [graph-agent]  expand_ownership

  ── Resolved entities ────────────────────────────────────
  'VODAFONE 2.'                  → 'VODAFONE 2.'  (exact=True)

  ── Steps ────────────────────────────────────────────────
  step_1  [graph-agent]  entity_lookup                        SUCCESS
       tools: entity_lookup
       └─ Found 10 company match(es) for 'VODAFONE 2.'. 

## Test 2 — Ownership + risk

Expected flow: `entity_lookup` → `expand_ownership` → `summarize_risk_for_company`

Expected planner: mode=investigate, 3 steps, graph-agent + risk-agent.

In [4]:
r2 = orchestrator.run(
    query="who owns VODAFONE 2. and is it risky?",
    user_id="analyst-304",
)
trace_ids.append(r2.trace_id)
show("Test 2 — Ownership + risk", r2)

# Risk findings
risk_step = next((s for s in r2.step_results if s.task == "summarize_risk_for_company"), None)
if risk_step and risk_step.success:
    for signal, key in [
        ("ownership_complexity_check", "ownership"),
        ("control_signal_check",       "control"),
        ("address_risk_check",         "address"),
        ("industry_context_check",     "industry"),
    ]:
        sig_data = risk_step.findings.get(signal) or {}
        level    = sig_data.get("risk_level", "n/a") if sig_data else "n/a"
        print(f"  {key:<10} risk: {level}")

[03/23/26 19:25:51] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=459290;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=224206;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:25:53] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=824272;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=320216;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:25:56] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=321305;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=577496;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


═════════════════════════════════════════════════════════════════
  Test 2 — Ownership + risk
═════════════════════════════════════════════════════════════════
  Query        : who owns VODAFONE 2. and is it risky?
  Mode         : investigate
  Success      : True
  Trace ID     : 032cef18-6b64-4b8a-8577-319fd69532d7

  ── Planner ──────────────────────────────────────────────
  Reason       : Combined ownership and risk query for VODAFONE 2. Resolving the canonical company name first, then walking the ownership chain and synthesising all four risk signals.
  Entity       : VODAFONE 2  (Company)
  Plan step    : step_1  [graph-agent]  entity_lookup
  Plan step    : step_2  [graph-agent]  expand_ownership
  Plan step    : step_3  [risk-agent]  summarize_risk_for_company

  ── Resolved entities ────────────────────────────────────
  'VODAFONE 2'                   → 'VODAFONE 2.'  (exact=False)

  ── Steps ────────────────────────────────────────────────
  step_1  [graph-agent]  entity_

## Test 3 — Trace replay

Uses the `trace_id` from Test 1 to replay an existing investigation.

Expected planner: mode=trace, 1 step, trace-agent retrieve_trace.

In [5]:
replay_id = r1.trace_id
print(f"Replaying trace: {replay_id}")

r3 = orchestrator.run(
    query=f"replay investigation trace {replay_id}",
    user_id="analyst-304",
)
trace_ids.append(r3.trace_id)
show("Test 3 — Trace replay", r3)

# Show events in the retrieved trace
retrieve_step = next(
    (s for s in r3.step_results
     if s.task in ("retrieve_trace", "retrieve_and_summarize_trace")),
    None,
)
if retrieve_step and retrieve_step.success:
    if retrieve_step.task == "retrieve_and_summarize_trace":
        trace_data = (
            (retrieve_step.findings.get("retrieve_and_summarize_trace") or {})
            .get("trace_data") or {}
        )
    else:
        trace_data = retrieve_step.findings.get("retrieve_trace") or {}
    events = trace_data.get("events", [])
    print(f"\nEvents in replayed trace: {len(events)}")
    for e in events:
        print(f"  [{e.get('event_type')}]  {e.get('tool_name','')}")

print(f"\nRetrieved trace ID : {r3.retrieved_trace_id}")
print(f"Matches replay_id  : {r3.retrieved_trace_id == replay_id}")

Replaying trace: 80097453-0625-40fe-9455-b05583e6de69


[03/23/26 19:25:58] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=954513;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=819856;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:26:01] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=245309;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=272638;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


═════════════════════════════════════════════════════════════════
  Test 3 — Trace replay
═════════════════════════════════════════════════════════════════
  Query        : replay investigation trace 80097453-0625-40fe-9455-b05583e6de69
  Mode         : trace
  Success      : True
  Trace ID     : 7f3a4d56-3ad6-4efe-8d94-b9e4a3878e25

  ── Planner ──────────────────────────────────────────────
  Reason       : User wants to retrieve and replay a specific prior investigation by trace ID, including a narrative summary.
  Entity       : 80097453-0625-40fe-9455-b05583e6de69  (Trace)
  Plan step    : step_1  [trace-agent]  retrieve_and_summarize_trace

  ── Steps ────────────────────────────────────────────────
  step_1  [trace-agent]  retrieve_and_summarize_trace         SUCCESS
       tools: retrieve_trace
       └─ The investigation examined VODAFONE 2. (company number 04083193) to assess benef…

  ── Final answer ─────────────────────────────────────────
  The investigation examined VO

## Test 3b — Latest trace by entity name (no trace ID needed)

Uses the canonical entity name from Test 1 to auto-retrieve and summarise
the most recent investigation without knowing the trace ID upfront.

Expected planner: mode=trace, 1 step, trace-agent `retrieve_latest_for_entity`.

In [6]:
entity_for_replay = (
    next(iter(r1.resolved_entities.values()), {}) or {}
).get("canonical_name") or r1.query
print(f"Entity : {entity_for_replay}")
print(f"Expected retrieved trace ID: {r1.trace_id}")

r3b = orchestrator.run(
    query=f"summarise the most recent investigation of {entity_for_replay}",
    user_id="analyst-304",
)
trace_ids.append(r3b.trace_id)
show("Test 3b — Latest trace by entity", r3b)

# Verify the correct trace was retrieved
latest_step = next(
    (s for s in r3b.step_results if s.task == "retrieve_latest_for_entity"),
    None,
)
if latest_step and latest_step.success:
    findings = latest_step.findings.get("retrieve_latest_for_entity") or {}
    print(f"\nTraces found for entity : {findings.get('traces_found')}")
    print(f"Latest trace ID         : {findings.get('latest_trace_id')}")

print(f"\nretrieved_trace_id      : {r3b.retrieved_trace_id}")
print(f"Matches r1.trace_id     : {r3b.retrieved_trace_id == r1.trace_id}")

Entity : VODAFONE 2.
Expected retrieved trace ID: 80097453-0625-40fe-9455-b05583e6de69


[03/23/26 19:26:02] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=484514;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=934865;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:26:05] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=417832;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=682513;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


═════════════════════════════════════════════════════════════════
  Test 3b — Latest trace by entity
═════════════════════════════════════════════════════════════════
  Query        : summarise the most recent investigation of VODAFONE 2.
  Mode         : trace
  Success      : True
  Trace ID     : 6e870ae1-dc16-462b-b7cd-c98d3e75468c

  ── Planner ──────────────────────────────────────────────
  Reason       : User is requesting the most recent investigation for a named entity without supplying a trace ID. Using retrieve_latest_for_entity to find and summarise the latest investigation automatically.
  Entity       : VODAFONE 2  (Company)
  Plan step    : step_1  [trace-agent]  retrieve_latest_for_entity

  ── Steps ────────────────────────────────────────────────
  step_1  [trace-agent]  retrieve_latest_for_entity           SUCCESS
       tools: find_traces_by_entity, retrieve_trace
       └─ The investigation examined VODAFONE 2 (company number 04083193), an active entit…

  ── Fin

## Test 4 — Unresolved entity

Entity does not exist in the graph. Expected behaviour:
- `resolve_entity` returns no match → entity gate closes
- `entity_lookup` still runs (shows zero candidates) → gate stays closed
- All subsequent entity-dependent steps are skipped (via entity gate)
- `errors` contains the unresolved entity message
- `final_answer` explains the failure

In [7]:
r4 = orchestrator.run(
    query="who owns ZZZQQQAAA FFFRRREEE BBBWWWYYY and is it risky?",
    user_id="analyst-304",
)
trace_ids.append(r4.trace_id)
show("Test 4 — Unresolved entity", r4)

# Verify all steps are skipped
skipped = [s for s in r4.step_results if s.skipped]
print(f"\nSkipped steps: {len(skipped)}")
for s in skipped:
    print(f"  {s.step_id}  {s.task}  — {s.skip_reason}")

assert len(skipped) == len(r4.step_results), (
    f"Expected all {len(r4.step_results)} steps skipped, got {len(skipped)}"
)
print("✓ All steps correctly skipped — entity not resolved")
print(f"\nFinal answer:\n{r4.final_answer}")

[03/23/26 19:26:07] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=439992;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=555157;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:26:09] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=990427;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=767674;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


═════════════════════════════════════════════════════════════════
  Test 4 — Unresolved entity
═════════════════════════════════════════════════════════════════
  Query        : who owns ZZZQQQAAA FFFRRREEE BBBWWWYYY and is it risky?
  Mode         : investigate
  Success      : False
  Trace ID     : 6e539fa5-c4ea-4748-85f4-005321b15439

  ── Planner ──────────────────────────────────────────────
  Reason       : Combined ownership and risk query. Resolving canonical name first, then walking the ownership chain and synthesising all risk signals.
  Entity       : ZZZQQQAAA FFFRRREEE BBBWWWYYY  (Company)
  Plan step    : step_1  [graph-agent]  entity_lookup
  Plan step    : step_2  [graph-agent]  expand_ownership
  Plan step    : step_3  [risk-agent]  summarize_risk_for_company

  ── Resolved entities ────────────────────────────────────
  'ZZZQQQAAA FFFRRREEE BBBWWWYYY' → NOT RESOLVED

  ── Steps ────────────────────────────────────────────────
  step_1  [graph-agent]  entity_lookup  

## Test 5 — Partial risk assessment (2 of 4 tools blocked)

Simulates a Kong MCP Gateway blocking `address_risk_check` and `industry_context_check`.

Expected behaviour:
- `success=True` — investigation continues with 2 available signals
- `findings["unavailable_tools"]` lists the 2 blocked tools
- 2 `STEP_FAILED` events in the trace (one per blocked tool)
- AI narrative synthesised from the 2 collected signals

In [8]:
from unittest.mock import patch
from src.domain.models import EventType, InvestigationTrace, ToolResult

BLOCKED = {"address_risk_check", "industry_context_check"}

_original_call_tool = mcp.call_tool  # capture before patch

def _selective_call_tool(tool_name, arguments):
    if tool_name in BLOCKED:
        return ToolResult(
            tool_name=tool_name, success=False,
            error="403 Forbidden", data=None,
            input=arguments, summary="", duration_ms=0,
        )
    return _original_call_tool(tool_name, arguments)  # call original, not patched

with patch.object(mcp, "call_tool", side_effect=_selective_call_tool):
    trace5 = InvestigationTrace(
        request_id=str(uuid.uuid4()),
        entity_name="VODAFONE LIMITED",
        user_id="test",
        mode="investigate",
    )
    r5 = risk_agent.run(
        "summarize_risk_for_company",
        {"company_name": "VODAFONE LIMITED"},
        trace5,
    )

assert r5.success, "Should succeed with 2/4 signals"
assert r5.findings.get("unavailable_tools") == sorted(BLOCKED), \
    f"Expected blocked tools in findings, got {r5.findings.get('unavailable_tools')}"
assert r5.findings.get("ownership_complexity_check") is not None
assert r5.findings.get("control_signal_check") is not None
assert r5.findings.get("address_risk_check") is None
assert r5.findings.get("industry_context_check") is None

failed_events = [e for e in trace5.events if e.event_type == EventType.STEP_FAILED]
assert len(failed_events) == 2, f"Expected 2 STEP_FAILED events, got {len(failed_events)}"

print(f"success={r5.success}")
print(f"unavailable_tools={r5.findings['unavailable_tools']}")
print(f"STEP_FAILED events in trace: {len(failed_events)}")
print(f"\nFinal answer:\n{r5.summary}")

[03/23/26 19:26:12] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=880330;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=251466;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

success=True
unavailable_tools=['address_risk_check', 'industry_context_check']
STEP_FAILED events in trace: 2

Final answer:
VODAFONE LIMITED presents a HIGH overall risk profile driven primarily by ownership complexity. The company exhibits a four-hop ownership chain with four distinct corporate entities and no identified individual ultimate beneficial owner, meeting the threshold for layered corporate structures that may obscure true beneficial ownership. While control mechanisms are confined to standard share-based arrangements (LOW risk) and address/industry context would require additional data to assess, the opaque ownership chain alone—particularly the absence of any natural-person UBO disclosure—warrants escalation for enhanced beneficial ownership verification under UK PSC rules. Risk level: HIGH.


## Comparison: all runs

In [9]:
runs = [
    ("ownership only",      r1),
    ("ownership + risk",    r2),
    ("trace replay",        r3),
    ("latest trace/entity", r3b),
    ("unresolved entity",   r4),
    ("partial risk 2/4",    r5),
]

print(f"{'Label':<24}  {'Mode':<13}  {'OK':>4}  {'Skipped':>7}  Trace ID")
print("-" * 84)
for label, r in runs:
    if hasattr(r, "step_results"):          # OrchestratorResult
        ok      = sum(1 for s in r.step_results if s.status == "success")
        total   = len(r.step_results)
        skipped = sum(1 for s in r.step_results if s.status == "skipped")
        mode    = r.mode
        tid     = r.trace_id[:16]
    else:                                   # AgentResult (direct agent call)
        ok      = 1 if r.success else 0
        total   = 1
        skipped = 0
        mode    = "agent-direct"
        tid     = r.trace.request_id[:16] if r.trace else "n/a"
    print(
        f"{label:<24}  {mode:<13}  {ok:>2}/{total:<2}  "
        f"{skipped:>7}  {tid}…"
    )

Label                     Mode             OK  Skipped  Trace ID
------------------------------------------------------------------------------------
ownership only            investigate     2/2         0  80097453-0625-40…
ownership + risk          investigate     3/3         0  032cef18-6b64-4b…
trace replay              trace           1/1         0  7f3a4d56-3ad6-4e…
latest trace/entity       trace           1/1         0  6e870ae1-dc16-46…
unresolved entity         investigate     0/3         3  6e539fa5-c4ea-47…
partial risk 2/4          agent-direct    1/1         0  4cd5db1a-227d-4f…


## Raw OrchestratorResult JSON

Full structured output for Test 2 (ownership + risk) — the richest result.

In [10]:
result_dict = r2.to_dict()

# Drop large findings dicts to keep output readable
for sr in result_dict["step_results"]:
    if sr["findings"]:
        sr["findings"] = {k: "<data>" for k in sr["findings"]}

print(json.dumps(result_dict, indent=2, default=str))

{
  "mode": "investigate",
  "query": "who owns VODAFONE 2. and is it risky?",
  "trace_id": "032cef18-6b64-4b8a-8577-319fd69532d7",
  "success": true,
  "planner_output": {
    "mode": "investigate",
    "reason": "Combined ownership and risk query for VODAFONE 2. Resolving the canonical company name first, then walking the ownership chain and synthesising all four risk signals.",
    "entities": [
      {
        "name": "VODAFONE 2",
        "type": "Company"
      }
    ],
    "plan": [
      {
        "step_id": "step_1",
        "agent": "graph-agent",
        "task": "entity_lookup",
        "parameters": {
          "name": "VODAFONE 2"
        }
      },
      {
        "step_id": "step_2",
        "agent": "graph-agent",
        "task": "expand_ownership",
        "parameters": {
          "company_name": "VODAFONE 2",
          "max_depth": 5
        }
      },
      {
        "step_id": "step_3",
        "agent": "risk-agent",
        "task": "summarize_risk_for_company",
 

## Verify traces in Neo4j

In [11]:
rows = trace_repo.list_traces(limit=10)
print(f"Recent traces in Neo4j: {len(rows)}")
print()
print(f"{'trace_id (first 16)':<20}  {'query':<35}  {'events':>6}")
print("-" * 65)
for t in rows:
    tid    = t.get("trace_id", "")[:16]
    query  = (t.get("query") or "")[:33]
    events = t.get("event_count", 0)
    print(f"{tid}…  {query:<35}  {events:>6}")

Recent traces in Neo4j: 8

trace_id (first 16)   query                                events
-----------------------------------------------------------------
6e539fa5-c4ea-47…  ZZZQQQAAA FFFRRREEE BBBWWWYYY             2
6e870ae1-dc16-46…  VODAFONE 2                                4
7f3a4d56-3ad6-4e…  80097453-0625-40fe-9455-b05583e6d         3
032cef18-6b64-4b…  VODAFONE 2                               10
80097453-0625-40…  VODAFONE 2.                               5
48b1e085-c991-4d…  ZZZZZ_NONEXISTENT_COMPANY_XYZ             2
6355a523-81f2-41…  Retrieve and summarise the invest         1
cb5b4cea-c7d9-40…  GLOBAL METALS UK LTD                     10


## Cleanup

In [12]:
for tid in trace_ids:
    if not tid:
        continue
    result = trace_repo.delete_trace(tid)
    print(f"Deleted {tid[:16]}…  found={result['found']}  events={result['events_deleted']}")

Deleted 80097453-0625-40…  found=True  events=5
Deleted 032cef18-6b64-4b…  found=True  events=10
Deleted 7f3a4d56-3ad6-4e…  found=True  events=3
Deleted 6e870ae1-dc16-46…  found=True  events=4
Deleted 6e539fa5-c4ea-47…  found=True  events=2


In [13]:
mcp.close()    # closes the MCP server's internal Neo4j connection
neo4j.close()  # closes the trace_service connection
print("Drivers closed")

Drivers closed
